# Aula 1 · Anotação, auditoria e partição dos dados

Notebook da Aula 1 do guia **Visão computacional aplicada com CVAT e YOLO** (github.com/tassid/pdi-agro).

Conjunto de dados: **WGISD**, Embrapa Wine Grape Instance Segmentation Dataset (Santos et al., 2020, *Computers and Electronics in Agriculture* 170, 105247). 300 fotos de 5 variedades de uva, caixas já no formato YOLO. Licença CC BY-NC 4.0.

| Seção | Bloco da aula |
|---|---|
| 1. Baixar o banco | 2.1 |
| 2. Ver as caixas e converter formatos | 1.2 |
| 3. Inferência com modelo pré-treinado, IoU e NMS à mão | 1.3 |
| 4. Auditar rótulos (do WGISD e do seu export do CVAT) | 1.5 |
| 5. Dividir por variedade e provar que não vazou | 1.5 |
| 6. Gerar a pasta no formato do Ultralytics | 1.5 |

Ambiente de execução → **GPU (T4)** só é necessário a partir da seção 3.

## 1. Obtenção do conjunto de dados

In [ ]:
# Baixa o WGISD (Embrapa): só as anotações e as fotos, sem histórico do git
import os, subprocess
if not os.path.exists("wgisd"):
    subprocess.run("git clone --depth 1 https://github.com/thsant/wgisd.git", shell=True, check=True)
import glob, random, shutil, yaml
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
fotos = sorted(glob.glob("wgisd/data/*.jpg"))
df = pd.DataFrame({"foto": fotos})
df["nome"] = df.foto.str.split("/").str[-1].str.replace(".jpg", "", regex=False)
df["variedade"] = df.nome.str[:3]
df["caixas"] = [sum(1 for l in open(f.replace(".jpg", ".txt")) if l.strip()) for f in df.foto]
df.groupby("variedade").agg(fotos=("nome", "size"), caixas=("caixas", "sum"))

## 2. Visualização das anotações e conversão de formatos

In [ ]:
def ler_yolo(foto):
    return np.array([list(map(float, l.split()[1:])) for l in open(foto.replace(".jpg", ".txt")) if l.strip()])

def yolo_para_xyxy(c, W, H):
    xc, yc, w, h = c
    return np.array([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H])

def desenhar(ax, foto, caixas=None, cor="yellow"):
    im = Image.open(foto); W, H = im.size; ax.imshow(im); ax.axis("off")
    for c in (ler_yolo(foto) if caixas is None else caixas):
        x1, y1, x2, y2 = yolo_para_xyxy(c, W, H)
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=cor, lw=2))

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (v, g) in zip(axes, df.groupby("variedade")):
    desenhar(ax, g.foto.iloc[0]); ax.set_title(v)
plt.show()

# conversão: a primeira caixa da primeira foto nos três formatos
foto = df.foto.iloc[0]; W, H = Image.open(foto).size; c = ler_yolo(foto)[0]
x1, y1, x2, y2 = yolo_para_xyxy(c, W, H)
print("YOLO :", c.round(4))
print("xyxy :", [round(v) for v in (x1, y1, x2, y2)])
print("COCO :", [round(x1), round(y1), round(x2 - x1), round(y2 - y1)])

### 2b. Distância de amostragem no solo (GSD) e resolução efetiva do objeto

In [ ]:
def gsd_cm(altura_m, pixel_um, focal_mm):
    return altura_m * pixel_um / focal_mm / 10          # cm por pixel

for h in [10, 20, 40, 80, 120]:
    g = gsd_cm(h, 2.4, 8.8)
    obj_px = 15 / g                                      # cacho de 15 cm
    print(f"{h:4d} m → GSD {g:.2f} cm/px | cacho com {obj_px:5.1f} px na foto | {obj_px * 640 / 5472:4.1f} px em imgsz 640")

## 3. Inferência com modelo pré-treinado, IoU e NMS

O COCO não contém a classe uva; registre as classes e confianças previstas pelo modelo.

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
coco = YOLO("yolo11n.pt")
r = coco.predict(df.foto.iloc[0], conf=0.1, verbose=False)[0]
print({coco.names[int(c)]: round(float(s), 2) for c, s in zip(r.boxes.cls, r.boxes.conf)})
r.show()

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

print(iou([100, 100, 200, 200], [150, 120, 250, 220]))   # confira com a conta do quadro: 0,25

def nms(caixas, scores, limiar=0.5):
    ordem = np.argsort(-np.array(scores)); manter = []
    for i in ordem:
        if all(iou(caixas[i], caixas[j]) < limiar for j in manter):
            manter.append(i)
    return manter

# detector "de brincadeira": cada caixa real vira 4 caixas tremidas com notas diferentes
rng = np.random.default_rng(0)
reais = [yolo_para_xyxy(c, W, H) for c in ler_yolo(foto)]
dets, notas = [], []
for g in reais:
    for k in range(4):
        dets.append(g + rng.normal(0, 15, 4)); notas.append(0.9 - 0.15 * k + rng.normal(0, 0.03))
for lim in [0.3, 0.5, 0.7]:
    print(f"NMS {lim}: {len(dets)} → {len(nms(dets, notas, lim))} caixas (reais: {len(reais)})")

## 4. Auditoria de rótulos

Execute a auditoria no WGISD e, em seguida, no conjunto exportado do CVAT (alterando `PASTA`).

In [ ]:
def auditar(rotulos, n_classes=1):
    problemas = []
    for f in rotulos:
        caixas = []
        for k, linha in enumerate(open(f)):
            if not linha.strip(): continue
            c, *v = linha.split(); v = list(map(float, v))
            if len(v) != 4: problemas.append((f, k, "linha sem 5 colunas")); continue
            if not 0 <= int(c) < n_classes: problemas.append((f, k, f"classe {c} não existe"))
            if any(x < 0 or x > 1 for x in v): problemas.append((f, k, "coordenada fora de [0, 1]"))
            if v[2] <= 0 or v[3] <= 0: problemas.append((f, k, "largura ou altura zero"))
            b = yolo_para_xyxy(v, 1, 1)
            if any(iou(b, o) > 0.9 for o in caixas): problemas.append((f, k, "duplicada"))
            caixas.append(b)
    return pd.DataFrame(problemas, columns=["arquivo", "linha", "problema"])

PASTA = "wgisd/data"
imgs = {os.path.basename(f)[:-4] for f in glob.glob(f"{PASTA}/*.jpg")}
rots = {os.path.basename(f)[:-4] for f in glob.glob(f"{PASTA}/*.txt")}
print("imagens sem rótulo:", len(imgs - rots), "| rótulos sem imagem:", len(rots - imgs))
auditar([f"{PASTA}/{n}.txt" for n in sorted(imgs & rots)])

**Verificação do auditor:** crie um arquivo de rótulos com inconsistências conhecidas e confirme que todas são identificadas.

In [ ]:
open("teste_errado.txt", "w").write("0 0.5 0.5 0.2 0.2\n0 0.51 0.5 0.2 0.2\n3 0.2 0.2 0.1 0.1\n0 0.95 0.5 0.2 0.0\n")
auditar(["teste_errado.txt"])

## 4b. Estatística descritiva do conjunto

Número de objetos por imagem, tamanho relativo e posição das caixas. Esses valores fundamentam a escolha de `imgsz`.

In [ ]:
linhas = []
for _, r in df.iterrows():
    for c in ler_yolo(r.foto):
        linhas.append({"variedade": r.variedade, "xc": c[0], "yc": c[1], "area_%": 100 * c[2] * c[3], "w/h": c[2] / c[3] * 2048 / 1365})
cx = pd.DataFrame(linhas)
print(df.groupby("variedade").caixas.describe()[["mean", "min", "max"]].round(1))
print(cx.groupby("variedade")["area_%"].median().round(2))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
cx["area_%"].hist(bins=40, ax=ax[0]); ax[0].set_title("área da caixa (% da foto)")
ax[1].hist2d(cx.xc, cx.yc, bins=12, cmap="magma"); ax[1].invert_yaxis(); ax[1].set_title("centros")
cx["w/h"].hist(bins=40, ax=ax[2]); ax[2].set_title("largura / altura")
plt.show()

### 4c. Concordância entre anotadores

Exporte do CVAT as anotações das mesmas imagens produzidas por cada integrante da dupla (pastas `A/` e `B/`).

In [ ]:
def concordancia(pasta_a, pasta_b, W=2048, H=1365, lim=0.5):
    total_iou, casadas, sem_par = [], 0, 0
    for fa in sorted(glob.glob(f"{pasta_a}/*.txt")):
        fb = os.path.join(pasta_b, os.path.basename(fa))
        A = [yolo_para_xyxy(list(map(float, l.split()[1:])), W, H) for l in open(fa) if l.strip()]
        B = [yolo_para_xyxy(list(map(float, l.split()[1:])), W, H) for l in open(fb) if l.strip()] if os.path.exists(fb) else []
        usados = set()
        for a in A:
            cand = [(iou(a, b), j) for j, b in enumerate(B) if j not in usados]
            v, j = max(cand, default=(0, -1))
            if v >= lim: usados.add(j); total_iou.append(v); casadas += 1
        sem_par += len(A) + len(B) - 2 * len(usados)
    return {"IoU médio": np.mean(total_iou) if total_iou else 0, "casadas": casadas, "sem par": sem_par}

# exemplo: compare o WGISD com ele mesmo deslocado (troque pelas pastas da dupla)
os.makedirs("A", exist_ok=True); os.makedirs("B", exist_ok=True)
for f in df.foto[:10]:
    n = os.path.basename(f).replace(".jpg", ".txt"); shutil.copy(f.replace(".jpg", ".txt"), f"A/{n}")
    with open(f"B/{n}", "w") as s:
        for l in open(f.replace(".jpg", ".txt")):
            c, *v = l.split(); v = np.array(list(map(float, v))) + rng.normal(0, 0.005, 4)
            s.write(c + " " + " ".join(f"{x:.5f}" for x in v) + "\n")
concordancia("A", "B")

## 5. Partição por variedade e verificação da ausência de vazamento

Cada variedade, cujas imagens provêm da mesma sessão e do mesmo vinhedo, é alocada integralmente a um único conjunto (ROBERTS et al., 2017).

In [ ]:
PAPEL = {"CDY": "train", "CFR": "train", "CSV": "train", "SVB": "val", "SYH": "test"}
df["conjunto"] = df.variedade.map(PAPEL)
print(df.groupby(["conjunto", "variedade"]).size())

# prova: nenhum grupo aparece em dois conjuntos
grupos = df.groupby("variedade").conjunto.nunique()
assert (grupos == 1).all(), grupos[grupos > 1]
print("ok: nenhum grupo em dois conjuntos")

# compare com uma divisão aleatória por foto: quantas variedades ficam espalhadas?
aleat = df.sample(frac=1, random_state=0).assign(c=lambda d: np.r_[["train"] * 210, ["val"] * 45, ["test"] * 45])
print("divisão aleatória: variedades presentes no treino E no teste =",
      len(set(aleat[aleat.c == "train"].variedade) & set(aleat[aleat.c == "test"].variedade)))

## 6. Estrutura de diretórios no formato Ultralytics

As imagens são reduzidas para no máximo 1280 pixels para adequar o tempo de treinamento à aula; como as coordenadas YOLO são normalizadas, as anotações permanecem válidas.

In [ ]:
SAIDA = "wgisd_yolo"
shutil.rmtree(SAIDA, ignore_errors=True)
for _, r in df.iterrows():
    for sub in ["images", "labels"]:
        os.makedirs(f"{SAIDA}/{sub}/{r.conjunto}", exist_ok=True)
    im = Image.open(r.foto).convert("RGB"); im.thumbnail((1280, 1280))
    im.save(f"{SAIDA}/images/{r.conjunto}/{r.nome}.jpg", quality=92)
    shutil.copy(r.foto.replace(".jpg", ".txt"), f"{SAIDA}/labels/{r.conjunto}/{r.nome}.txt")
yaml.safe_dump({"path": os.path.abspath(SAIDA), "train": "images/train", "val": "images/val",
                "test": "images/test", "names": {0: "grape"}}, open(f"{SAIDA}/data.yaml", "w"))
print(open(f"{SAIDA}/data.yaml").read())

## Exercícios

1. Altere `PAPEL` para usar Syrah na validação e Sauvignon Blanc no teste. Compare as estatísticas descritivas dos conjuntos.
2. Realize uma partição aleatória por imagem e uma por variedade. Quantas imagens de cada variedade ocorrem no teste em cada caso?
3. Implemente a detecção de caixas duplicadas entre arquivos de anotadores distintos e meça a concordância da dupla.
4. Com os valores da seção 2b, determine a altura de voo para que um cacho de 15 cm ocupe pelo menos 16 pixels com `imgsz=1280`.

## Referências

ROBERTS, D. R. et al. Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. Ecography, v. 40, n. 8, p. 913-929, 2017.

SANTOS, T. T. et al. Grape detection, segmentation, and tracking using deep neural networks and three-dimensional association. Computers and Electronics in Agriculture, v. 170, 105247, 2020.